## Executive Summary
Write a short (200 words max) summary of what the report contains and what you have discovered.

## Introduction
For this assignment, I have chosen the World Bank Group's Health, Nutrition and Population Statistics dataset. The dataset holds information gathered from a variety of national and internationl sources, covering several areas of public health and population development. These include but are not limited to, infectious disease, noncommunicable diseases, immunisations, healthcare resources and their uses, nutrition, population changes, reproductive health, along with access to clean water and sanitation and HIV/AIDS statistics.

I chose this dataset as public health and pathology is a major interest of mine, and with such a wide range of indicators available, i believe there are opportunities to explore how health and population statistics vary accross different sections of the world. I am particularly interested in seeing patterns between countries and looking at how certain health indicators change over time.

Trhoughout this report, i will be investigating the structure and quality of the dataset, explore its contents, and use descriptive analytics to answer questions about the information it provides. I hope to identify some interesting trends and gain a better understanding of what the data can tell us about global health.

## Data Cleaning
In this section I will take care of data cleaning, this will use a combination of pandas and other forms of programatical data cleaning with python.



Below we will begin by importing pandas and loading one of our csv's for further manilupation.

In [1]:
#imports the pandas functionality
import pandas as pd

Next we will view the dataset contents

In [2]:
#opens the csv to be used
country_df = pd.read_csv("HNP_StatsCountry.csv")

#creating a working copy, hella bad experience with this in the past, like not having one
clean_country = country_df.copy()

display(clean_country.head())

#next we will inspect the dataset
print("Data dimensions:", clean_country.shape)

print("\nData types:")
print(clean_country.dtypes)

print("\nMissing values:")
print(clean_country.isnull().sum())

print("\nDuplicate records:")
print(clean_country.duplicated().sum())


,Country Code,Short Name,Table Name,Long Name,2-alpha code,Currency Unit,Special Notes,Region,Income Group,WB-2 code,...,Government Accounting concept,IMF data dissemination standard,Latest population census,Latest household survey,Source of most recent Income and expenditure data,Vital registration complete,Latest agricultural census,Latest industrial data,Latest trade data,Unnamed: 30
0,ABW,Aruba,Aruba,Aruba,AW,Aruban florin,NaN,Latin America & Caribbean,High income,AW,...,NaN,Enhanced General Data Dissemination System (e-...,2020 (expected),NaN,NaN,Yes,NaN,NaN,2018.0,NaN
1,AFE,Africa Eastern and Southern,Africa Eastern and Southern,Africa Eastern and Southern,ZH,NaN,"26 countries, stretching from the Red Sea in t...",NaN,NaN,ZH,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AFG,Afghanistan,Afghanistan,Islamic State of Afghanistan,AF,Afghan afghani,The reporting period for national accounts dat...,South Asia,Low income,AF,...,Consolidated central government,Enhanced General Data Dissemination System (e-...,1979,"Demographic and Health Survey, 2015","Integrated household survey (IHS), 2016/17",NaN,NaN,NaN,2018.0,NaN
3,AFW,Africa Western and Central,Africa Western and Central,Africa Western and Central,ZI,NaN,"22 countries, stretching from the westernmost ...",NaN,NaN,ZI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AGO,Angola,Angola,People's Republic of Angola,AO,Angolan kwanza,The World Bank systematically assesses the app...,Sub-Saharan Africa,Lower middle income,AO,...,Budgetary central government,Enhanced General Data Dissemination System (e-...,2014,"Demographic and Health Survey, 2015/16","Integrated household survey (IHS), 2008/09",NaN,NaN,NaN,2018.0,NaN


Data dimensions: (265, 31)

Data types:
Country Code                                             str
Short Name                                               str
Table Name                                               str
Long Name                                                str
2-alpha code                                             str
Currency Unit                                            str
Special Notes                                            str
Region                                                   str
Income Group                                             str
WB-2 code                                                str
National accounts base year                              str
National accounts reference year                     float64
SNA price valuation                                      str
Lending category                                         str
Other groups                                             str
System of National Accounts                  

Next we will remove completely empty columns as they provide 0 usefule information for us to analyze.

In [3]:
#removes all empty collumns
clean_country = clean_country.dropna(
    axis = 1,
    how = "all"
)

print("Updated dimensions:", clean_country.shape)


Updated dimensions: (265, 28)


Now we will be removing duplicate records as they hold the potential to distort statistical analysis.

In [4]:
#this cleans and prints out the data before and after it was cleaned, even if there is nothing to clean
#its good practice
print("Duplicates before cleaning:",clean_country.duplicated().sum())

clean_country = clean_country.drop_duplicates()

print("Duplicates after cleaning:",clean_country.duplicated().sum())


Duplicates before cleaning: 0
Duplicates after cleaning: 0


Next we will remove unnecessary white space as this helps maintain standardised text values and prevents problems when comparing or mathcing records.

In [13]:
#this removes unnecessary white space
for column in clean_country.columns:

    if pd.api.types.is_string_dtype(clean_country[column]):
        clean_country[column] = (
            clean_country[column].str.strip()
        )


Now following all our cleaning and checking, we will inspect the dataset again for any remaining missing values that may have not been removed in our initial removal, this is because our other actions can create other issing values

In [6]:
print(clean_country.isnull().sum())

Country Code                                           0
Short Name                                             0
Table Name                                             0
Long Name                                              0
2-alpha code                                           2
Currency Unit                                         48
Special Notes                                        139
Region                                                48
Income Group                                          49
WB-2 code                                              1
National accounts base year                           57
National accounts reference year                     192
SNA price valuation                                   58
Lending category                                     121
Other groups                                         206
System of National Accounts                           57
Balance of Payments Manual in use                     66
External debt Reporting status 

And finally, after all our initial cleaning procedures have been completed, we will export the cleaned data as a new CSV. This allows the initial unclean dataset to exist for the sake of comparison and allows us to use all our cleaned data.

In [7]:
#this saves all our changes to a new dataset
clean_country.to_csv(
    "cleaned_HNP_StatsCountry.csv",
    index = False
)


To help with further cleaning i will get some automation going, even more so that python is advertised as being good for that.

In [8]:
#this function automates all our previous data cleaning steps to be applied to 
#all the other sets and a make our work run smoother
def basic_cleaning(df):
    #create working copy
    df = df.copy()

    #remove completely empty columns
    df = df.dropna(axis = 1, how = "all")

    #remove duplicate rows
    df = df.drop_duplicates()

    #Remove unnecessary whitespace
    for column in df.columns:

        if pd.api.types.is_string_dtype(df[column]):
            df[column] = df[column].str.strip()
    return df

Sooooo, next step is to clean the other datasets

In [9]:
#first we are going to set up a dictionary to call upon all the other data

files = {
    "CountrySeries": "HNP_StatsCountry-Series.csv",
    "Data": "HNP_StatsData.csv",
    "FootNote": "HNP_StatsFootNote.csv",
    "Series": "HNP_StatsSeries.csv",
    "SeriesTime": "HNP_StatsSeries-Time.csv"
}

datasets = {}

for name, filename in files.items():
    datasets[name] = pd.read_csv(filename)

    print(f"{name}: {datasets[name].shape}")
#this loads all CSVs into one dictionary called datasets
#to this we can apply our cleaning function 

cleaned_datasets = {}

for name, df in datasets.items():
    print(f"\n--- {name} ---")
    print("Before:", df.shape)

    cleaned_df = basic_cleaning(df)
    print("After:", cleaned_df.shape)
    cleaned_datasets[name] = cleaned_df

#this applies our cleaning function to the rest of the data sets removing entierly empty collumns, duplicate rows 
#and unnecessary white space
#it also keeps the original data frames untouched

#next we will document the changes made and save them to new cleaned files
for name, df in datasets.items():

    cleaned_df = cleaned_datasets[name]
    print(f"\n--- {name} ---")
    print("Rows Removed:", len(df) - len(cleaned_df))
    print("Columns Removed:", df.shape[1] - cleaned_df.shape[1])
    print("Missing values before:", df.isnull().sum().sum())
    print("Missing values after:", cleaned_df.isnull().sum().sum())

#this saves all our clean data to new clean CSV files

for name, df in cleaned_datasets.items():
    df.to_csv(f"Cleaned_HNP_Stats{name}.csv",index = False)


CountrySeries: (3394, 4)
Data: (117838, 67)
FootNote: (529470, 5)
Series: (443, 21)
SeriesTime: (465, 4)

--- CountrySeries ---
Before: (3394, 4)
After: (3394, 3)

--- Data ---
Before: (117838, 67)
After: (117838, 66)

--- FootNote ---
Before: (529470, 5)
After: (529470, 4)

--- Series ---
Before: (443, 21)
After: (443, 14)

--- SeriesTime ---
Before: (465, 4)
After: (465, 3)

--- CountrySeries ---
Rows Removed: 0
Columns Removed: 1
Missing values before: 3394
Missing values after: 0

--- Data ---
Rows Removed: 0
Columns Removed: 1
Missing values before: 4127330
Missing values after: 4009492

--- FootNote ---
Rows Removed: 0
Columns Removed: 1
Missing values before: 529470
Missing values after: 0

--- Series ---
Rows Removed: 0
Columns Removed: 7
Missing values before: 5158
Missing values after: 2057

--- SeriesTime ---
Rows Removed: 0
Columns Removed: 1
Missing values before: 465
Missing values after: 0


This concludes our cleaning, the data is now lacking useless or poorly formatted data cells and rows, which also explains why there still is data returning missing values, as some values may stay empty but still provide useful data in the values that are populated.

## Exploratory Data Analysis
To being, we will take a quick peak at the size of each table so we can better unnderstan what we are working with.

In [10]:
#this prints all our data in order to allow us to make some observations and see what we are working with
for name, df in cleaned_datasets.items():
    print(f"\n---{name}---")
    print("Shape:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing rows:", df.isnull().sum().sum())
    display(df.head(3))


---CountrySeries---
Shape: (3394, 3)
Duplicate rows: 0
Missing rows: 0


,CountryCode,SeriesCode,DESCRIPTION
0,ABW,SP.DYN.LE00.MA.IN,Data source: United Nations World Population P...
1,ABW,SP.DYN.LE00.FE.IN,Data source: United Nations World Population P...
2,ABW,SP.DYN.CDRT.IN,Data source: United Nations World Population P...



---Data---
Shape: (117838, 66)
Duplicate rows: 0
Missing rows: 4009492


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
0,Africa Eastern and Southern,AFE,"Adolescent fertility rate (births per 1,000 wo...",SP.ADO.TFRT,140.120535,140.594564,141.128109,141.32344,141.50816,141.668896,...,105.256166,103.391461,101.541782,99.719799,97.916601,96.146492,94.695223,93.231944,91.744712,NaN
1,Africa Eastern and Southern,AFE,Adults (ages 15+) and children (0-14 years) li...,SH.HIV.TOTL,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Africa Eastern and Southern,AFE,Adults (ages 15+) and children (ages 0-14) new...,SH.HIV.INCD.TL,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



---FootNote---
Shape: (529470, 4)
Duplicate rows: 0
Missing rows: 0


,CountryCode,SeriesCode,Year,DESCRIPTION
0,ABW,SH.TBS.INCD,YR2020,Uncertainty bound is 0.92-1.2
1,ABW,SH.TBS.INCD,YR2019,Uncertainty bound is 1.9-2.5
2,ABW,SH.TBS.INCD,YR2018,Uncertainty bound is 4.6-6.3



---Series---
Shape: (443, 14)
Duplicate rows: 0
Missing rows: 2057


,Series Code,Topic,Indicator Name,Short definition,Long definition,Periodicity,Aggregation method,Limitations and exceptions,Notes from original source,General comments,Source,Statistical concept and methodology,Development relevance,License Type
0,HD.HCI.OVRL,Public Sector: Policy & institutions,Human capital index (HCI) (scale 0-1),NaN,The HCI calculates the contributions of health...,NaN,NaN,NaN,NaN,NaN,World Bank staff calculations based on the met...,NaN,NaN,CC BY-4.0
1,HD.HCI.OVRL.FE,Public Sector: Policy & institutions,"Human capital index (HCI), female (scale 0-1)",NaN,The HCI calculates the contributions of health...,NaN,NaN,NaN,NaN,NaN,World Bank staff calculations based on the met...,NaN,NaN,CC BY-4.0
2,HD.HCI.OVRL.LB,Public Sector: Policy & institutions,"Human capital index (HCI), lower bound (scale ...",NaN,The HCI lower bound reflects uncertainty in th...,NaN,NaN,NaN,NaN,NaN,World Bank staff calculations based on the met...,NaN,NaN,CC BY-4.0



---SeriesTime---
Shape: (465, 3)
Duplicate rows: 0
Missing rows: 0


,SeriesCode,Year,DESCRIPTION
0,SP.ADO.TFRT,YR1960,Interpolated using data for 1957 and 1962.
1,SP.DYN.TO65.MA.ZS,YR1960,Interpolated using data for 1957 and 1962.
2,SP.DYN.TO65.FE.ZS,YR1960,Interpolated using data for 1957 and 1962.


So, from a supreficial skim trough all the data we can see that each set has a different role, that being, Data holds the indicators by country and year, Series holds descriptions of the indicators and CountrySeries, FootNote, SeriesTime all provide source notes, footnotes and time-related notes.

Now we will explore how much data there is for each country-indicator combination. This will aid with identifying indicators with consistent coverage accross the years and choose suitable ones for further analysis.

In [11]:
data = cleaned_datasets["Data"]

#id the columns containing yearly values
year_columns = [col for col in cleaned_datasets["Data"].columns if str(col).isdigit()]

#count reported country-year values for each indicator
reported_by_indicator = (cleaned_datasets["Data"].groupby("Indicator Name")[year_columns].count().sum(axis = 1).sort_values(ascending = False))

#here we cover indicators with the highes percentage cover
country_count = data["Country Code"].nunique()
possible_values = country_count * len(year_columns)

coverage_by_indicator = (data.groupby("Indicator Name")[year_columns].count().sum(axis = 1).div(possible_values).mul(100).sort_values(ascending = False))

#prints all our results
display(reported_by_indicator.sort_values(ascending = False).head(10).to_frame("Reported year values"))
display(coverage_by_indicator.head(10).round(1).to_frame("Coverage (%)")) 

#here we will cover indicators with the lowest percentage indicator
#this is all to show a balance and range of data for us to better analyse 
display(reported_by_indicator.sort_values().head(10).to_frame("Reported country-year values"))


,Reported year values
Indicator Name,
"Population, total",16149
Rural population (% of total population),16058
Urban population (% of total population),16058
Urban population,16025
Rural population,16025
Population growth (annual %),15886
Urban population growth (annual %),15765
Rural population growth (annual %),15264
"Birth rate, crude (per 1,000 people)",14897


,Coverage (%)
Indicator Name,
"Population, total",97.9
Rural population (% of total population),97.4
Urban population (% of total population),97.4
Urban population,97.2
Rural population,97.2
Population growth (annual %),96.3
Urban population growth (annual %),95.6
Rural population growth (annual %),92.6
"Birth rate, crude (per 1,000 people)",90.3


,Reported country-year values
Indicator Name,
Urban poverty headcount ratio at national poverty lines (% of urban population),0
Rural poverty headcount ratio at national poverty lines (% of rural population),0
Number of people pushed further below the $3.20 ($ 2011 PPP) poverty line by out-of-pocket health care expenditure,60
Number of people pushed further below the $1.90 ($ 2011 PPP) poverty line by out-of-pocket health care expenditure,60
Number of people spending more than 25% of household consumption or income on out-of-pocket health care expenditure,60
Number of people pushed further below the 60% median consumption poverty line by out-of-pocket health care expenditure,60
Number of people pushed below the $3.20 ($ 2011 PPP) poverty line by out-of-pocket health care expenditure,60
Number of people pushed below the 60% median consumption poverty line by out-of-pocket health care expenditure,60
Number of people spending more than 10% of household consumption or income on out-of-pocket health care expenditure,60


Now me personally i would say that is enough analysis to allow us to make some observations, so for our observations:

1) Population indicators have compratively high coverage.
Population total gold 16149 reported country-year values, around 97.9% of the rough maximum used in our analysis. Rural and urban population measures also have coverage above 97%. This makes them promising candidates for further analysis. Althought percentages from our analysis are estimates rather than 100% true percentages.

2) Some indicators have every limited or no recorded data
The least reported results include indicators with 0 reported values and others with 60. This shows us that data availability varies considerablly accross the datasets. Analysis using these indicators may need a narrower time period or selected countries.

3) The main table combines many indicators accross a long time span, but the data is unevenly reported
The sets contain 62 yearly columns, from 1960 to 2021, and 4009492 missing cells (in the Data table specifically). This mins that we are offered broad historical coverag but missingness should be checked before comparing countries or years, as some gaps will greatly affect the results of the data.

That concludes our Exploratory Data Analysis, and we will further explore our observations along with questions that arrise from them in the Descriptive Analytics section.


## Descriptive Analytics
## 1) How has total population changed over time ?


In [12]:
data = cleaned_datasets["Data"]

population = data[data["Indicator Code"] == "SP.POP.TOTL"]

print("Rows for total population:",len(population))
display(population[["Country Name", "Country Code", "Indicator Name"] + year_columns].head())

Rows for total population: 266


,Country Name,Country Code,Indicator Name,1960,1961,1962,1963,1964,1965,1966,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
329,Africa Eastern and Southern,AFE,"Population, total",130836765.0,134159786.0,137614644.0,141202036.0,144920186.0,148769974.0,152752671.0,...,547482863.0,562601578.0,578075373.0,593871847.0,609978946.0,626392880.0,643090131.0,660046272.0,677243299.0,694665117.0
772,Africa Western and Central,AFW,"Population, total",96396419.0,98407221.0,100506960.0,102691339.0,104953470.0,107289875.0,109701811.0,...,370243017.0,380437896.0,390882979.0,401586651.0,412551299.0,423769930.0,435229381.0,446911598.0,458803476.0,470898870.0
1215,Arab World,ARB,"Population, total",92197715.0,94724540.0,97334438.0,100034191.0,102832792.0,105736428.0,108758634.0,...,371437642.0,379696477.0,387899835.0,396028301.0,404042892.0,411942825.0,419851989.0,427870273.0,436080728.0,444517783.0
1658,Caribbean small states,CSS,"Population, total",4194711.0,4274052.0,4353623.0,4432240.0,4508189.0,4580382.0,4648353.0,...,7072640.0,7123315.0,7173443.0,7222197.0,7269385.0,7314956.0,7358929.0,7401389.0,7442291.0,7481631.0
2101,Central Europe and the Baltics,CEB,"Population, total",91401764.0,92232738.0,93009498.0,93840016.0,94715795.0,95440988.0,96146336.0,...,103935318.0,103713726.0,103496179.0,103257886.0,102994278.0,102740078.0,102538451.0,102398494.0,102172351.0,101669618.0


## 2) How have urban and rural population patterns changed over time ?

## 3) Is population growth associated with changes in life expectancy or infant mortality ?

## Suggestions
1) One way of enriching the data would be to combine it with external data, for example; income levels (as GDP possibly), education (primarily illiteracy) or further healthcare data (such as spending and funding, which is already partly covered but could recieve further complements from external data). This could give further insight into health and population differencesbetween countries, an example would be a comparison between healthcare spensing and life expectancy or infant moratlity rate. However any external data (sources could include CDC, WHO, NHS, etc.) would need to be formatted to the country codes given by the World Bank Group.

2) A further suggestion, is to focus on indicators with strong data coverage, such as total population and its spread between urban and rural areas, and this could then be analysed alongside indicators with lower percentage coverage, as a possible approach, but there should be a treshold at which certain indicators are removed if they dont provide enough data, but that should also be done carefully as some countries do not provide enough data in the first place.

3) Another suggestion would be to use statystical analysis to investigate relationships over time. For example a correlation analysis a can explore whether changes in urbanistion have any realtion to overall population growth or health changes. A time series model could be used to examine trends and give possible future values.However these methods will need careful interpretation as correlation does not prove caustions and predictions may be unreliable where dat is sparse or historical patterns change.

So, overall, adding relevant external data and applying further statystical analysis could provide a more detailed understanding of the relationship between health, population and development. These steps will build on the current dataset while accounting for its uneven coverage and missing values.

## Bibliography
- https://datacatalog.worldbank.org/search/dataset/0037652/Health-Nutrition-and-Population-Statistics